<a href="https://colab.research.google.com/github/anonymous-rflego-ae/mobicom26_ae/blob/main/ae/RF_LEGO_AE_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RF-LEGO — Artifact Evaluation (AE)

This notebook runs the RF-LEGO artifact-evaluation workflow end to end. It demonstrates four things:

1. **Artifact functionality**: installation, model initialization, a forward/backward pass, and shipped-weight inference for each of the three RF-LEGO modules.
2. **Result evaluation**: side-by-side, quantitative module-level results comparing RF-LEGO with the corresponding classical signal processing baselines.
3. **AE benchmark**: modality-specific held-out `.npz` files under `ae_data/`, using model-ready real-world inputs and ground truth.
4. **Cascadability check** (related to paper Sec. 5.2.2): comparing classical front ends→CFAR with RF-LEGO range/Doppler FT→Detector and angle Beamformer→Detector pipelines.

The three module-level result groups:

| Module | Baseline | Metric |
|------|---|---|
| Frequency Transform | Bluestein FFT | PSLR / PAPR improvement (dB) |
| Beamformer | LASSO Beamformer | angle-MAE reduction (%) |
| Detector | CA-CFAR (4 training / 2 guard per side | exact-bin DR at method-native fixed thresholds |

### Evaluation path

The notebook uses the prepared modality `.npz` files and shipped pretrained weights, then evaluates, plots, and runs the reduced cascadability check (roughly **3 minutes** on Colab CPU).

Metrics are computed over each discovered `ae_data/*.npz` file. The 50-frame mmWave Beamformer benchmark contains prepared real-measurement frames spanning angles from -60° to +60°.

In [ ]:
# --- repository source ---

# --- install package + AE dependencies ---
import os
import subprocess
import sys

OWNER = "anonymous-rflego-ae"      # GitHub owner/org; edit before publishing
REPO = "mobicom26_ae"    # GitHub repo name
BRANCH = "main"

def sh(*a):
    return subprocess.run(list(a), check=True)

if not os.path.isdir("ae"):
    sh("git", "clone", "--depth", "1", "-b", BRANCH, f"https://github.com/{OWNER}/{REPO}.git", "rflego_repo")
    if os.path.isdir("rflego_repo/ae"):
        os.chdir("rflego_repo")
assert os.path.isdir("ae"), "Could not locate the repo (set OWNER/REPO above)."

sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".")
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "ae/requirements.txt")
print("repo root:", os.getcwd())


In [ ]:
# --- deterministic setup + environment snapshot (seed = 42) ---
import json
import sys

sys.path.insert(0, "ae/scripts")
import _common as C  # noqa: E402

C.ensure_dirs()
C.set_determinism(42)
env = C.environment_info()
C.write_json(C.ENV_DIR / "environment.json", env)
print(json.dumps(env, indent=2))


In [ ]:
# --- locate the prepared AE data files and pretrained weights ---
def have_pretrained_weights():
    return all((C.WEIGHTS_DIR / f"{m}.pt").exists() for m in C.MODULES)

benchmarks = C.discover_benchmarks()
if not benchmarks:
    raise FileNotFoundError("Missing AE data files under ae_data/*.npz")

print("benchmark files:")
for spec in benchmarks:
    print(f"  {spec.id:20s} {spec.modality_label:7s} {spec.task_label:12s} -> {spec.path.relative_to(C.REPO_ROOT)}")

needed_modules = sorted({spec.module for spec in benchmarks})
missing_weights = [str((C.WEIGHTS_DIR / f"{m}.pt").relative_to(C.REPO_ROOT)) for m in needed_modules if not (C.WEIGHTS_DIR / f"{m}.pt").exists()]
if missing_weights:
    raise FileNotFoundError("Missing pretrained weights: " + ", ".join(missing_weights))

print("pretrained weights present:", have_pretrained_weights())


In [ ]:
# --- functional smoke test: build each model, run forward + backward ---
import torch

from rflego.modules import BeamformerModel, DetectorModel, FrequencyTransformModel

C.set_determinism(42)
g = torch.Generator().manual_seed(42)

# FT
ft = FrequencyTransformModel(C.build_model_config("ft", C.load_yaml(C.module_config_path("ft"))))
xr = torch.randn(8, 256, generator=g)
xi = torch.randn(8, 256, generator=g)
yr, yi = ft(xr, xi)
loss = (yr**2 + yi**2).mean()
loss.backward()
gnorm = sum(p.grad.abs().sum() for p in ft.parameters() if p.grad is not None)
print(f"FT         out={tuple(yr.shape)}  grad_finite={torch.isfinite(gnorm).item()}")

# Beamformer
bf = BeamformerModel(C.build_model_config("beamformer", C.load_yaml(C.module_config_path("beamformer"))))
y = torch.randn(8, 8, dtype=torch.complex64)
A = torch.randn(8, 8, 121, dtype=torch.complex64)
z = bf(y, A)
loss = z.abs().mean()
loss.backward()
gnorm = sum(p.grad.abs().sum() for p in bf.parameters() if p.grad is not None)
print(f"Beamformer out={tuple(z.shape)}  grad_finite={torch.isfinite(gnorm).item()}")

# Detector (sequence-first [L, B])
det = DetectorModel(C.build_model_config("detector", C.load_yaml(C.module_config_path("detector"))))
x = torch.randn(128, 8)
logits = det(x)
loss = logits.mean()
loss.backward()
gnorm = sum(p.grad.abs().sum() for p in det.parameters() if p.grad is not None)
transition = det.layers[0].transition
a_grad = transition.A.grad.abs().sum().item()
b_grad = transition.B.grad.abs().sum().item()
print(f"Detector   out={tuple(logits.shape)}  grad_finite={torch.isfinite(gnorm).item()}")
print(f"Detector A/B trainable={transition.A.requires_grad and transition.B.requires_grad}; first-step grad L1: A={a_grad:.6g}, B={b_grad:.6g}")
assert a_grad > 0 and b_grad > 0, "Detector A/B must receive sample-conditioned gradients on the first backward pass"
print("Detector raw-first input path OK: first-step A/B gradients are nonzero.")
print("\nSmoke test OK: all three modules run forward + backward.")


In [ ]:
# --- select shipped pretrained weights ---

ACTIVE_WEIGHTS_DIR = C.WEIGHTS_DIR
missing = [str(ACTIVE_WEIGHTS_DIR / f"{m}.pt") for m in C.MODULES if not (ACTIVE_WEIGHTS_DIR / f"{m}.pt").exists()]
assert not missing, "Missing pretrained weights: " + ", ".join(missing)
print("Using shipped pretrained weights:", ACTIVE_WEIGHTS_DIR)


In [ ]:
# --- evaluate all modality/task files ---
subprocess.run([
    sys.executable,
    "ae/scripts/evaluate.py",
    "--module",
    "all",
    "--weights-dir",
    str(ACTIVE_WEIGHTS_DIR),
], check=True)
summary = C.read_json(C.METRICS_DIR / "summary.json")
print("evaluated datasets:", ", ".join(row["id"] for row in summary["rows"]))


In [ ]:
# --- render one fixed visualization per dataset ---
from IPython.display import Image, display

for existing_fig in C.FIGURES_DIR.glob("result_*.png"):
    existing_fig.unlink()
subprocess.run([
    sys.executable,
    "ae/scripts/plot_reproduction.py",
], check=True)

for row in C.read_json(C.METRICS_DIR / "summary.json")["rows"]:
    path = C.FIGURES_DIR / f"result_{row['id']}.png"
    print(path.name)
    display(Image(filename=str(path)))


In [ ]:
# --- final modality/module summary table ---
summary = C.read_json(C.METRICS_DIR / "summary.json")
rows = [[row["modality"], row["module"], row["samples"], row["results"], row["analysis"]] for row in summary["rows"]]
hdr = ["modality", "module", "samples", "results", "analysis"]
w = [max(len(str(r[i])) for r in ([hdr] + rows)) for i in range(len(hdr))]
def fmt(r): return "  ".join(str(c).ljust(w[i]) for i, c in enumerate(r))
print(fmt(hdr))
print("-" * (sum(w) + 2 * (len(hdr) - 1)))
for r in rows:
    print(fmt(r))


In [ ]:
# --- detailed scalar metrics ---
for row in C.read_json(C.METRICS_DIR / "summary.json")["rows"]:
    m = C.read_json(C.METRICS_DIR / f"{row['id']}.json")
    print(f"\n{row['modality']} / {row['module']} ({row['samples']} samples)")
    print("  baseline:", m["baseline"])
    print("  RF-LEGO :", m["rflego"])
    if "improvement" in m:
        print("  improvement:", m["improvement"])
    else:
        print("  comparison:", m["comparison_note"])


## Reduced cascadability check (related to paper Sec. 5.2.2)

RF-LEGO modules are designed to compose. The next cell reports strict exact-bin Detection Rate (DR) at fixed operating points in a common regime of **FAR at the `10^-3` order of magnitude**:

- range / Doppler : front-end **FT -> Detector**   (classical: Bluestein FFT -> CA-CFAR)
- angle           : **Beamformer → Detector**     (classical: LASSO → CA-CFAR)

All learned paths use per-profile min-max normalization to `[0,1]`. Range and Doppler retain 4 training and 2 guard cells per side. Angle alone uses the fixed experimental setting of 18 training and 10 guard cells per side. Both paths are read out at fixed, retained scalar cutoffs that are constants of the artifact. RF-LEGO Detector has no native `P_FA` parameter; its logit cutoff defines the binary readout but does not change the trained Detector. We therefore describe the paths as **FAR-regime aligned** at the same order of magnitude.

**Measured with the corrected 1300-step Detector.** CA-CFAR DR is 0.780 for range, 0.400 for Doppler, and 0.560 for angle. RF-LEGO DR is 0.880, 0.960, and 0.760, respectively. Only the exact ground-truth center bin can count as a hit; adjacent bins are negatives. **Scope vs. the paper.** All paths operate in the same `10^-3`-order low-FAR regime. These values must not be presented as an exact reproduction of paper Sec. 5.2.2.

In [ ]:
# --- cascadability: RF-LEGO vs classical front ends (mmWave) ---
subprocess.run([
    sys.executable,
    "ae/scripts/cascade.py",
    "--weights-dir",
    str(ACTIVE_WEIGHTS_DIR),
], check=True)

cascade = C.read_json(C.METRICS_DIR / "cascade.json")
hdr = ["pipeline", "CA window", "CA-CFAR DR", "RF-LEGO DR"]
rows = []
for r in cascade["rows"]:
    rows.append([
        r["pipeline"],
        f"{r['ca_cfar_n_train_per_side']}/{r['ca_cfar_n_guard_per_side']}",
        f"{r['dr_classical']:.3f}",
        f"{r['dr_rflego']:.3f}",
    ])
w = [max(len(str(r[i])) for r in ([hdr] + rows)) for i in range(len(hdr))]
def _fmt(r): return "  ".join(str(c).ljust(w[i]) for i, c in enumerate(r))
print("RF-LEGO vs classical front ends (mmWave):\n")
print("Note: both paths are read out at fixed retained cutoffs in the same 10^-3-order FAR regime.\n")
print(_fmt(hdr))
print("-" * (sum(w) + 2 * (len(hdr) - 1)))
for r in rows:
    print(_fmt(r))